# 01 · Preprocessing — from raw data to a `DolceSet`

Every model in `dolcestat` eats a **`DolceSet`**: a small container holding your
feature matrix `X`, your target `y`, and a memory of the transformations you've
applied. This notebook builds one, step by step:

1. **Load** a table and name the target.
2. **Inspect** what you've got.
3. **Encode** categorical columns into numbers.
4. **Scale** features onto comparable ranges.

By the end you'll have a clean `DolceSet` ready to hand to any model.

## 1 · Loading data

`DolceSet` loads from a plain Python **dict** (`load_from_dict`) or a **Polars
DataFrame** (`load_from_polars_dataframe`). Either way you pass `target_col` —
the column to predict — and it separates the features from the target.

Our toy dataset: apartment **price** (the target) from **size**, **rooms**, and
a categorical **district**.

In [1]:
from dolcestat.preprocessing import DolceSet

raw = {
    "size_m2":  [55, 80, 120, 42, 95, 150, 60, 200, 70, 110],
    "rooms":    [2, 3, 4, 1, 3, 5, 2, 6, 2, 4],
    "district": ["center", "suburb", "center", "suburb", "center",
                 "suburb", "center", "suburb", "center", "suburb"],
    "price_k":  [180, 210, 340, 120, 260, 400, 175, 520, 190, 300],
}

data = DolceSet()
data.load_from_dict(raw, target_col="price_k")

## 2 · Inspecting the set

A handful of accessors show what the `DolceSet` holds:

- `get_features()` / `get_target()` — the feature names and the target name
- `get_colnames()` — every column, target included
- `get_column(name)` — one column as a NumPy array
- `.X` / `.y` — the feature matrix and target vector models actually use

In [2]:
print("features:", data.get_features())
print("target:  ", data.get_target())
print("X shape: ", data.X.shape, " y shape:", data.y.shape)
data.get_column("size_m2")

features: ['size_m2', 'rooms', 'district']
target:   price_k
X shape:  (10, 3)  y shape: (10,)


array([ 55,  80, 120,  42,  95, 150,  60, 200,  70, 110])

## 3 · Encoding categoricals

`district` is text — models need numbers. `one_hot_encode` turns one categorical
column into 0/1 indicator columns, one per category. It uses **drop-first**
encoding (with *k* categories you get *k − 1* columns), which avoids the
dummy-variable trap where the indicator columns become perfectly collinear.

Watch the feature list change — and note the target `price_k` stays out of it.

In [3]:
print("before:", data.get_features())
data.one_hot_encode("district")
print("after: ", data.get_features())

before: ['size_m2', 'rooms', 'district']
after:  ['size_m2', 'rooms', 'district_suburb']


## 4 · Scaling

`size_m2` runs into the hundreds while `rooms` stays single-digit. Distance-based
models (like KNN) and gradient-based optimizers behave far better when features
share a comparable range. `dolcestat` offers two strategies:

- **`"standardize"`** — subtract the mean, divide by the std (→ mean 0, std 1).
- **`"min-max"`** — rescale onto the `[0, 1]` interval.

Call `scale(strategy)` to transform *every* feature, or
`scale(strategy, "column")` to target one. Statistics are learned from the data
automatically, and `scaling_info` records exactly what was applied.

In [4]:
data.scale("standardize", "size_m2")
data.scale("min-max", "rooms")

for step in data.scaling_info["details"]:
    print(step["strategy"], "→", step["columns"])

data.X[:3]

standardize → ['size_m2']
min-max → ['rooms']


array([[-0.88838837,  0.2       ,  0.        ],
       [-0.37427473,  0.4       ,  1.        ],
       [ 0.44830709,  0.6       ,  0.        ]])

### Applying training statistics to new data

To avoid **leakage**, scale your test data with the *training* statistics, not
its own. Every `scale` call records the stats it used; pass them back through the
`mean`/`std` (or `min`/`max`) keyword arguments:

In [5]:
train = DolceSet()
train.load_from_dict({"x": [1.0, 2.0, 3.0, 4.0], "y": [0, 0, 1, 1]}, target_col="y")
train.scale("standardize", "x")
stats = train.scaling_info["details"][0]["stats"]

test = DolceSet()
test.load_from_dict({"x": [5.0, 6.0], "y": [1, 1]}, target_col="y")
test.scale("standardize", "x", mean=stats["mean"], std=stats["std"])
test.X.flatten()

array([1.93649167, 2.71108834])

## Recap

You now have a `DolceSet` that is loaded, encoded, and scaled — everything a
model needs. Next up: [`02_linear_regression`](02_linear_regression.ipynb),
where we hand a set like this to your first model.